# <center> Deep Learning Coursework </center>
## <center> Audio Text Transformer Implementation </center>

Explaination of Audio Text Transformer

Imports required

In [ ]:
import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision import transforms

In [ ]:
%run audio-text_transformer.py --learning-rate 1e-4 --seed 42

In [ ]:
weights = torch.load(
    "transformer_weights.pth",
    map_location="cpu",
    weights_only=True
)

for name, tensor in weights.items():
    print(f"Layer: {name}")
    print(f"Shape: {tensor.shape}")
    print(f"Weights:\n{tensor}")
    print("-" * 50)

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_root = Path("/data/COMS30017")

# Load validation features
val_audio, val_text, val_labels = load_features(
    dataset_root, "val"
)

# Create validation dataset
val_dataset = AudioTextDataset(
    val_audio, val_text, val_labels, training=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

# Get feature dimensions from the dataset
first_audio, first_text, _ = val_dataset[0]

# Initialise your actual model
model = AudioTextTransformer(
    audio_dim=first_audio.shape[-1],
    text_dim=first_text.shape[-1],
    d_model=MODEL_DIM,
    num_heads=4,
    mlp_dim=TRANSFORMER_MLP_DIM,
    dropout=DROPOUT
)

# Load trained model weights
model.load_state_dict(
    torch.load(
        "transformer_weights.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

model = model.to(DEVICE)

# Evaluation setup
criterion = nn.CrossEntropyLoss()
model.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, text, labels in val_loader:

        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        # Get model predictions
        logits = model(audio, text)

        # Calculate loss
        loss = criterion(logits, labels)
        total_loss += loss.item()

        # Get predicted classes
        predictions = logits.argmax(dim=-1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = np.mean(
    np.array(all_preds) == np.array(all_labels)
)

average_loss = total_loss / len(val_loader)

print(f"Validation Loss: {average_loss:.5f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Total Samples: {len(all_labels)}")